In [1]:
!pip install yt-dlp pandas

# Scraper tahap 1 Menggunakan keyword
Output: json platform, judul, link, creator, tanggal upload, views, like, jumlah komentar

In [ ]:
import yt_dlp
import json
import os
import re
from datetime import datetime

def bersihkan_nama_folder(nama):
    nama_bersih = re.sub(r'[\\/*?:"<>|]', "", nama)
    return nama_bersih.strip()

def scrape_youtube_metadata(keyword, max_results=5):
    ydl_opts = {
        'quiet': True,              
        'extract_flat': False,      
        'ignoreerrors': True,       
        'no_warnings': True
    }

    search_query = f"ytsearch{max_results}:{keyword}"
    timestamp = datetime.now().isoformat()
    data_list = []
    
    folder_induk = f"data/youtube_{keyword.replace(' ', '_')}_assets"

    print(f"\n{'='*50}")
    print(f"Mencari {max_results} video untuk keyword: '{keyword}'...")
    print(f"{'='*50}")

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            result = ydl.extract_info(search_query, download=False)
            
            if 'entries' in result:
                for index, video in enumerate(result['entries'], start=1):
                    if not video: continue
                    
                    raw_date = video.get('upload_date', '')
                    formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}" if len(raw_date) == 8 else raw_date

                    judul_asli = video.get('title')
                    judul_aman = bersihkan_nama_folder(judul_asli)
                    path_video = os.path.join(folder_induk, judul_aman).replace("\\", "/")
                    
                    # 1. Mengekstrak ID Unik Video
                    video_id = video.get('id')

                    item = {
                        "scraped_at": timestamp,
                        "platform": "YouTube",
                        "keyword_pencarian": keyword,
                        "video_id": video_id,  # <-- ID Unik disimpan di sini
                        "judul": judul_asli,
                        "related_link": video.get('webpage_url') or f"https://www.youtube.com/watch?v={video_id}",
                        "creator": video.get('uploader'),
                        "tanggal_upload": formatted_date,
                        "views": video.get('view_count', 0),
                        "likes": video.get('like_count', 0),
                        "jumlah_komentar": video.get('comment_count', 0),
                        "lokasi_penyimpanan": path_video
                    }
                    data_list.append(item)
                    print(f"[{index}/{max_results}] ✔️ Berhasil: {judul_asli[:40]}...")
        except Exception as e:
            print(f"Error saat scraping: {e}")

    return data_list

# ==========================================
# EKSEKUSI TAHAP 1
# ==========================================
daftar_keyword = ["kucing", "anjing"] # Bisa ditambah
jumlah_target_per_keyword = 15

os.makedirs("data", exist_ok=True)

for kw in daftar_keyword:
    hasil = scrape_youtube_metadata(kw, max_results=jumlah_target_per_keyword)

    if hasil:
        filename = f"data/youtube_{kw.replace(' ', '_')}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(hasil, f, indent=4, ensure_ascii=False)
        print(f"✅ Metadata '{kw}' disimpan di: {filename}")


Mencari 15 video teratas untuk keyword: 'balap kuda'...



ERROR: [youtube] rxFdyPlXsFs: This video is not available


[1/15] ✔️ Berhasil: FULL RACE PACUAN KUDA PANGANDARAN 2025...
[2/15] ✔️ Berhasil: Kentucky Derby 2023 (FULL RACE) | NBC Sports...
[3/15] ✔️ Berhasil: WHR's Ten Best Races of 2024! Ft. DO DEUCE and VIA...
[4/15] ✔️ Berhasil: kuda Nakal Paling Belakang 🐎 Juara 1. indahnya Dun...
[5/15] ✔️ Berhasil: KUDA SEHARGA LAMBORGINI! 25 Jenis Kuda Tercantik d...
[6/15] ✔️ Berhasil: Indahnya Pacu Kuda ‼️ PERHATIKAN KUDA PUTIH INI. M...
[7/15] ✔️ Berhasil: Dubai Gold Cup Sponsored by Al Tayer Motors...
[8/15] ✔️ Berhasil: RACE REPLAY | KELAS TERBUKA - 2000 M | JATENG DERB...
[9/15] ✔️ Berhasil: RACE REPLAY | KELAS 3 TAHUN DERBY "INDONESIA DERBY...
[10/15] ✔️ Berhasil: Dubai World Cup 2025 - HIT SHOW (4YO+ WFA G1) Grou...
[11/15] ✔️ Berhasil: 2025 Whitney Stakes: Champions Fierceness, Sierra ...
[12/15] ✔️ Berhasil: BERNASIB SAMA SEBAGAI YANG TERBUANG PASANGAN KUDA ...
[13/15] ✔️ Berhasil: INDONESIA'S HORSE RACING: JATENG DERBY 2026...
[14/15] ✔️ Berhasil: KUMPULAN INSIDENT & TRAGEDI PACUAN KUDA...

✅

# Scarping Komentar berdasarkan link pada json sebelumnya
Output: json, Judul, komentar, nama akun yang komentar

In [5]:
import json
import yt_dlp
import os

file_video_sumber = "data/youtube_balap_kuda.json"

def scrape_komentar_ke_folder(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    ydl_opts_comments = {
        'quiet': True,
        'extract_flat': False,
        'skip_download': True,    
        'getcomments': True,      
        'extractor_args': {'youtube': {'comment_sort': ['top'], 'max_comments': ['20,all,all']}},
        'ignoreerrors': True,
        'no_warnings': True
    }

    print(f"Memulai scraping komentar dan menyebarnya ke folder masing-masing...\n")

    with yt_dlp.YoutubeDL(ydl_opts_comments) as ydl:
        for index, video in enumerate(video_data, start=1):
            url = video.get('related_link')
            judul = video.get('judul')
            lokasi_folder = video.get('lokasi_penyimpanan')
            
            print(f"[{index}/{len(video_data)}] Memproses: {judul[:40]}...")
            
            # 1. Pastikan foldernya dibuat
            os.makedirs(lokasi_folder, exist_ok=True)
            
            try:
                info = ydl.extract_info(url, download=False)
                comments = info.get('comments', [])
                
                if not comments:
                    print("  -> ⚠️ Tidak ada komentar.")
                    continue
                
                hasil_komentar_video_ini = []
                
                for c in comments:
                    id_komentar = c.get('id')
                    id_parent = c.get('parent')
                    
                    # Logika Reply
                    tipe_komentar = "Komentar Utama" if (id_parent == 'root' or id_parent is None) else "Balasan"

                    komentar_item = {
                        "id_komentar": id_komentar,
                        "id_parent": id_parent,
                        "tipe": tipe_komentar,
                        "nama_channel": c.get('author'),
                        "isi_komentar": c.get('text')
                    }
                    hasil_komentar_video_ini.append(komentar_item)
                
                # 2. Simpan file komentar.json khusus di dalam folder video tersebut
                file_komentar = os.path.join(lokasi_folder, "komentar.json")
                with open(file_komentar, "w", encoding="utf-8") as f:
                    json.dump(hasil_komentar_video_ini, f, indent=4, ensure_ascii=False)
                
                print(f"  -> ✔️ Tersimpan {len(comments)} komentar di: {file_komentar}")
                
            except Exception as e:
                print(f"  -> ❌ Error: {e}")

# ==========================================
# EKSEKUSI TAHAP 2
# ==========================================
if os.path.exists(file_video_sumber):
    scrape_komentar_ke_folder(file_video_sumber)
    print("\n✅ Proses ekstraksi komentar ke masing-masing folder selesai!")
else:
    print(f"File {file_video_sumber} tidak ditemukan.")

Memulai scraping komentar dan menyebarnya ke folder masing-masing...

[1/14] Memproses: FULL RACE PACUAN KUDA PANGANDARAN 2025...
  -> ✔️ Tersimpan 21 komentar di: data/youtube_balap_kuda_assets/FULL RACE PACUAN KUDA PANGANDARAN 2025\komentar.json
[2/14] Memproses: Kentucky Derby 2023 (FULL RACE) | NBC Sp...
  -> ✔️ Tersimpan 1405 komentar di: data/youtube_balap_kuda_assets/Kentucky Derby 2023 (FULL RACE)  NBC Sports\komentar.json
[3/14] Memproses: WHR's Ten Best Races of 2024! Ft. DO DEU...
  -> ✔️ Tersimpan 36 komentar di: data/youtube_balap_kuda_assets/WHR's Ten Best Races of 2024! Ft. DO DEUCE and VIA SISTINA!\komentar.json
[4/14] Memproses: kuda Nakal Paling Belakang 🐎 Juara 1. in...
  -> ✔️ Tersimpan 227 komentar di: data/youtube_balap_kuda_assets/kuda Nakal Paling Belakang 🐎 Juara 1. indahnya Dunia Pacuan Kuda\komentar.json
[5/14] Memproses: KUDA SEHARGA LAMBORGINI! 25 Jenis Kuda T...
  -> ✔️ Tersimpan 146 komentar di: data/youtube_balap_kuda_assets/KUDA SEHARGA LAMBORGINI! 25 J

# Scarping download video berdasarkan link hasil scarp keyword

In [9]:
import json
import yt_dlp
import os

file_video_sumber = "data/youtube_balap_kuda.json"

def download_video_ke_folder(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    print(f"Memulai proses download untuk {len(video_data)} video...\n")

    for index, video in enumerate(video_data, start=1):
        url = video.get('related_link')
        judul = video.get('judul')
        lokasi_folder = video.get('lokasi_penyimpanan')
        
        # Pastikan folder sudah ada (jaga-jaga jika skip tahap 2)
        os.makedirs(lokasi_folder, exist_ok=True)
        
        print(f"\n{'='*60}")
        print(f"[{index}/{len(video_data)}] Mengunduh ke: {lokasi_folder}")
        print(f"{'='*60}")
        
        # Konfigurasi diset dinamis agar 'outtmpl' mengarah ke folder yang tepat
        ydl_opts_download = {
            'format': 'best', # Ambil yang gambar+suara sudah menyatu
            'outtmpl': f'{lokasi_folder}/video_lengkap.%(ext)s', # Nama file diseragamkan
            'quiet': False, 
            'ignoreerrors': True,
            'no_warnings': True
        }
        
        with yt_dlp.YoutubeDL(ydl_opts_download) as ydl:
            try:
                ydl.download([url])
            except Exception as e:
                print(f"❌ Gagal mengunduh {judul}: {e}")

# ==========================================
# EKSEKUSI TAHAP 3
# ==========================================
if os.path.exists(file_video_sumber):
    download_video_ke_folder(file_video_sumber)
    print(f"\n✅ Proses unduh selesai! Semua file telah masuk ke foldernya masing-masing.")
else:
    print(f"File {file_video_sumber} tidak ditemukan.")

Memulai proses download untuk 14 video...


[1/14] Mengunduh ke: data/youtube_balap_kuda_assets/FULL RACE PACUAN KUDA PANGANDARAN 2025
[youtube] Extracting URL: https://www.youtube.com/watch?v=qCiI30IVsaw
[youtube] qCiI30IVsaw: Downloading webpage
[youtube] qCiI30IVsaw: Downloading android vr player API JSON
[info] qCiI30IVsaw: Downloading 1 format(s): 18
[download] Destination: data\youtube_balap_kuda_assets\FULL RACE PACUAN KUDA PANGANDARAN 2025\video_lengkap.mp4
[download] 100% of  259.23MiB in 00:00:22 at 11.77MiB/s     

[2/14] Mengunduh ke: data/youtube_balap_kuda_assets/Kentucky Derby 2023 (FULL RACE)  NBC Sports
[youtube] Extracting URL: https://www.youtube.com/watch?v=aUDgaN6iHFc
[youtube] aUDgaN6iHFc: Downloading webpage
[youtube] aUDgaN6iHFc: Downloading android vr player API JSON
[info] aUDgaN6iHFc: Downloading 1 format(s): 18
[download] Destination: data\youtube_balap_kuda_assets\Kentucky Derby 2023 (FULL RACE)  NBC Sports\video_lengkap.mp4
[download] 100% of   23.91MiB in

# FINAL GABUNGAN

In [ ]:
import yt_dlp
import json
import os
import re
from datetime import datetime

# ==========================================
# FUNGSI 1: PEMBERSIH NAMA FOLDER
# ==========================================
def bersihkan_nama_folder(nama):
    nama_bersih = re.sub(r'[\\/*?:"<>|]', "", nama)
    return nama_bersih.strip()

# ==========================================
# FUNGSI 2: SCRAPING METADATA & DESKRIPSI (TAHAP 1)
# ==========================================
def scrape_youtube_metadata(keyword, max_results=5):
    ydl_opts = {
        'quiet': True,              
        'extract_flat': False,      
        'ignoreerrors': True,       
        'no_warnings': True
    }

    search_query = f"ytsearch{max_results}:{keyword}"
    timestamp = datetime.now().isoformat()
    data_list = []
    
    folder_induk = f"data/youtube_{keyword.replace(' ', '_')}_assets"
    os.makedirs("data", exist_ok=True)

    print(f"\n{'='*60}")
    print(f"🎬 TAHAP 1: MENCARI {max_results} VIDEO UNTUK '{keyword}'")
    print(f"{'='*60}")

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            result = ydl.extract_info(search_query, download=False)
            
            if 'entries' in result:
                for index, video in enumerate(result['entries'], start=1):
                    if not video: continue
                    
                    raw_date = video.get('upload_date', '')
                    formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}" if len(raw_date) == 8 else raw_date

                    judul_asli = video.get('title')
                    judul_aman = bersihkan_nama_folder(judul_asli)
                    video_id = video.get('id')
                    
                    # Membuat path spesifik untuk video ini
                    path_video = os.path.join(folder_induk, judul_aman).replace("\\", "/")
                    os.makedirs(path_video, exist_ok=True) # Langsung buat foldernya di sini

                    # Struktur Data Metadata Lengkap (Termasuk Deskripsi)
                    item = {
                        "scraped_at": timestamp,
                        "platform": "YouTube",
                        "keyword_pencarian": keyword,
                        "video_id": video_id,
                        "judul": judul_asli,
                        "deskripsi_video": video.get('description', ''), # <-- DESKRIPSI DITARIK DI SINI
                        "related_link": video.get('webpage_url') or f"https://www.youtube.com/watch?v={video_id}",
                        "creator": video.get('uploader'),
                        "tanggal_upload": formatted_date,
                        "views": video.get('view_count', 0),
                        "likes": video.get('like_count', 0),
                        "jumlah_komentar": video.get('comment_count', 0),
                        "lokasi_penyimpanan": path_video
                    }
                    data_list.append(item)
                    
                    # Simpan detail metadata ini ke dalam foldernya sendiri
                    file_metadata_lokal = f"{path_video}/metadata.json"
                    with open(file_metadata_lokal, "w", encoding="utf-8") as f:
                        json.dump(item, f, indent=4, ensure_ascii=False)

                    print(f"[{index}/{max_results}] ✔️ Folder & Metadata Dibuat: {judul_asli[:30]}...")
        except Exception as e:
            print(f"❌ Error saat scraping metadata: {e}")

    # Simpan indeks global
    if data_list:
        filename = f"data/youtube_{keyword.replace(' ', '_')}_index.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data_list, f, indent=4, ensure_ascii=False)
        print(f"✅ Indeks keseluruhan disimpan di: {filename}")
        return filename
    else:
        return None

# ==========================================
# FUNGSI 3: SCRAPING KOMENTAR (TAHAP 2)
# ==========================================
def scrape_komentar_ke_folder(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    ydl_opts_comments = {
        'quiet': True,
        'extract_flat': False,
        'skip_download': True,    
        'getcomments': True,      
        'extractor_args': {'youtube': {'comment_sort': ['top'], 'max_comments': ['20,all,all']}},
        'ignoreerrors': True,
        'no_warnings': True
    }

    print(f"\n💬 TAHAP 2: EKSTRAKSI KOMENTAR")
    
    with yt_dlp.YoutubeDL(ydl_opts_comments) as ydl:
        for index, video in enumerate(video_data, start=1):
            video_id = video.get('video_id') 
            judul = video.get('judul')
            lokasi_folder = video.get('lokasi_penyimpanan')
            
            print(f"[{index}/{len(video_data)}] Menarik komentar: {judul[:30]}...")
            
            try:
                info = ydl.extract_info(video_id, download=False)
                comments = info.get('comments', [])
                
                if not comments:
                    print("  -> ⚠️ Tidak ada komentar.")
                    continue
                
                hasil_komentar_video_ini = []
                for c in comments:
                    id_parent = c.get('parent')
                    tipe_komentar = "Komentar Utama" if (id_parent == 'root' or id_parent is None) else "Balasan"

                    komentar_item = {
                        "video_id": video_id,
                        "id_komentar": c.get('id'),
                        "id_parent": id_parent,
                        "tipe": tipe_komentar,
                        "nama_channel": c.get('author'),
                        "isi_komentar": c.get('text')
                    }
                    hasil_komentar_video_ini.append(komentar_item)
                
                # Disimpan langsung ke dalam folder video terkait
                file_komentar = os.path.join(lokasi_folder, "komentar.json")
                with open(file_komentar, "w", encoding="utf-8") as f:
                    json.dump(hasil_komentar_video_ini, f, indent=4, ensure_ascii=False)
                
                print(f"  -> ✔️ Tersimpan {len(comments)} komentar.")
            except Exception as e:
                print(f"  -> ❌ Error komentar: {e}")

# ==========================================
# FUNGSI 4: DOWNLOAD VIDEO (TAHAP 3)
# ==========================================
def download_video_ke_folder(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    print(f"\n⬇️ TAHAP 3: UNDUH VIDEO FISIK")

    for index, video in enumerate(video_data, start=1):
        url = video.get('related_link')
        judul = video.get('judul')
        lokasi_folder = video.get('lokasi_penyimpanan')
        
        print(f"[{index}/{len(video_data)}] Mengunduh video: {judul[:30]}...")
        
        ydl_opts_download = {
            'format': 'best', 
            'outtmpl': f'{lokasi_folder}/video_lengkap.%(ext)s', 
            'quiet': False, 
            'ignoreerrors': True,
            'no_warnings': True
        }
        
        with yt_dlp.YoutubeDL(ydl_opts_download) as ydl:
            try:
                ydl.download([url])
            except Exception as e:
                print(f"❌ Gagal mengunduh: {e}")

# ==========================================
# BLOK EKSEKUSI UTAMA (PIPELINE OTOMATIS)
# ==========================================
if __name__ == "__main__":
    # 1. Output dari Model Anda (Daftar Keyword) dimasukkan ke sini
    daftar_keyword_dari_model = ["penipu report"] 
    
    # 2. Tentukan jumlah video per keyword
    jumlah_target_per_keyword = 2 

    print("🚀 MEMULAI PIPELINE CRAWLER OTOMATIS...")
    
    for kw in daftar_keyword_dari_model:
        file_json_indeks = scrape_youtube_metadata(kw, max_results=jumlah_target_per_keyword)
        
        if file_json_indeks:
            scrape_komentar_ke_folder(file_json_indeks)
            download_video_ke_folder(file_json_indeks)
            print(f"\n🎉 PIPELINE SELESAI UNTUK KEYWORD: '{kw}'\n")

    print("\n🏁 SEMUA PROSES CRAWLING TELAH SELESAI!")